In [1]:
import pandas as pd
import re

# Read the list of filenames from the configuration file
with open('../file_list.txt', 'r', encoding='utf-8') as config_file:
    file_names = config_file.read().splitlines()

# Regex pattern to match the data format
pattern = r'\[(.*?)\] (.*?): (.*)'

# Initialize an empty list to store parsed data
datalist = []
stream_count = 0
# Iterate over each specified file
for file in file_names:
    full_path = f"../data/{file}"
    with open(full_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            match = re.match(pattern, line)
            if match:
                date, user, message = match.groups()
                datalist.append([date, user, message,stream_count])
    stream_count = stream_count + 1

# Create a DataFrame from the parsed data
data = pd.DataFrame(datalist, columns=["date", "user", "message","stream"])


In [2]:
data = data[data["user"] != "StreamElements"]
data = data[data["user"] != "Fossabot"]

In [3]:

data['date'] = pd.to_datetime(data['date'])



In [4]:
data["user"] = data["user"].replace("Banties1g", "banties_x")
data["user"] = data["user"].replace("banties1g", "banties_x")
data["user"] = data["user"].replace("chili_poe", "chili_con_bacon")
data["user"] = data["user"].replace("CHILI_POE", "chili_con_bacon")
data["user"] = data["user"].replace("Chili_poe", "chili_con_bacon")
data["user"] = data["user"].replace("chili_conbacon", "chili_con_bacon")
data["user"] = data["user"].replace("Wirelesss_", "W1r3lesss")
data["user"] = data["user"].replace("treklul", "trek44_")
data["user"] = data["user"].replace("ttrek_", "trek44_")
data["user"] = data["user"].replace("trek_x", "trek44_")
data["user"] = data["user"].replace("TriplesingleJ", "TripleSingleJames")
data["user"] = data["user"].replace("uwu_cougar", "uuccugr")
data["user"] = data["user"].replace("uuccugr_","uuccugr")
data["user"] = data["user"].replace("StanIV4_", "stan_iv4")
data["user"] = data["user"].replace("Muuskie2", "Muuskie")
data["user"] = data["user"].replace("nishad_more1311", "nishad13")
data["user"] = data["user"].replace("softarballt", "softarr")
data["user"] = data["user"].replace("softarballtt23", "softarr")
data["user"] = data["user"].replace("lajosbarnabas", "lajoss__")
data["user"] = data["user"].replace("Bonkwiththefunk", "bonk67")
data["user"] = data["user"].replace("qfishyy11", "bonk67")

In [5]:
from collections import defaultdict

# Get all unique usernames
unique_users = data['user'].unique()

# Create a mapping from lowercase username to all variants

user_variants = defaultdict(set)
for user in unique_users:
    user_variants[user.lower()].add(user)

# Find usernames with different capitalization
duplicate_users = {k: v for k, v in user_variants.items() if len(v) > 1}

In [6]:
# Create a mapping from all variants to the canonical (sorted first) variant
variant_map = {}
for variants in duplicate_users.values():
    sorted_variants = sorted(variants)
    canonical = sorted_variants[0]
    for v in variants:
        variant_map[v] = canonical

# Replace usernames in 'user' column
data['user'] = data['user'].apply(lambda u: variant_map.get(u, u))

In [7]:
# Count the number of messages per user
message_counts = data.groupby("user")["message"].count()

# Filter users with 25 or more messages
users_with_25_or_more = message_counts[message_counts >= 25].index

# Filter the original DataFrame to keep only these users
data = data[data["user"].isin(users_with_25_or_more)]

In [8]:
from collections import defaultdict

# Get all unique usernames
unique_users = data['user'].unique()

# Create a mapping from lowercase username to all variants

user_variants = defaultdict(set)
for user in unique_users:
    user_variants[user.lower()].add(user)

# Find usernames with different capitalization
duplicate_users = {k: v for k, v in user_variants.items() if len(v) > 1}

In [9]:
# Create a mapping from all variants to the canonical (sorted first) variant
variant_map = {}
for variants in duplicate_users.values():
    sorted_variants = sorted(variants)
    canonical = sorted_variants[0]
    for v in variants:
        variant_map[v] = canonical

# Replace usernames in 'user' column
data['user'] = data['user'].apply(lambda u: variant_map.get(u, u))

In [10]:
# Convert date to datetime format
data["date"] = pd.to_datetime(data["date"])



In [11]:
# Truncate datetime to just the day (removing time)
data["day"] = data["date"].dt.date  # Extract only the date part



In [12]:
# Group by 'day' and 'user' and calculate the message count per day per user
data["message_count"] = 1  # Assign 1 for each message to count them
daily_counts = data.groupby(["day", "user"])["message_count"].count().reset_index()



In [13]:
# Pivot the table to create a user-wise table for each day
pivot_table = daily_counts.pivot(index="day", columns="user", values="message_count").fillna(0)


In [14]:

# Add a cumulative sum for each user across the days
cumulative_pivot = pivot_table.cumsum()


In [15]:

# Print the result
print(cumulative_pivot)

user        0000000emirburak0320  00cad  00skysea00  00yopop  010justwatch  \
day                                                                          
2024-05-01                   0.0    0.0         0.0      0.0           0.0   
2024-05-02                   0.0    0.0         0.0      0.0           0.0   
2024-05-03                   0.0    0.0         0.0      0.0           0.0   
2024-05-04                   0.0    0.0         0.0      0.0           0.0   
2024-05-05                   0.0    0.0         0.0      0.0           0.0   
...                          ...    ...         ...      ...           ...   
2026-03-05                  49.0   55.0        27.0     55.0          57.0   
2026-03-06                  49.0   55.0        27.0     55.0          57.0   
2026-03-07                  49.0   55.0        27.0     55.0          57.0   
2026-03-08                  49.0   55.0        27.0     55.0          57.0   
2026-03-09                  49.0   55.0        27.0     55.0    

In [16]:
pivot_data_cleaned_transposed = cumulative_pivot.T
pivot_data_cleaned_transposed = cumulative_pivot.T.reset_index()

In [17]:
pivot_data_cleaned_transposed.head(5)
pivot_data_cleaned_transposed.tail(5)

day,user,2024-05-01,2024-05-02,2024-05-03,2024-05-04,2024-05-05,2024-05-06,2024-05-07,2024-05-09,2024-05-10,...,2026-02-27,2026-02-28,2026-03-01,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09
10568,빵댕이ㅋ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,74.0,75.0,77.0,77.0,78.0,78.0,79.0,79.0,79.0,79.0
10569,쌍베님사랑합니다,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0
10570,안톤958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0
10571,알래스카해달,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6723.0,6723.0,6723.0,6723.0,6742.0,6742.0,6744.0,6776.0,6804.0,6804.0
10572,엘레레레ㅔ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0,36.0


In [18]:
# --- Step 1 & 2: Calculate rank and identify top 20 users ---

# Calculate rank for each date (column) in descending order.
# The user with the highest cumulative sum gets rank 1.
ranked_df = pivot_data_cleaned_transposed.rank(axis=0, ascending=False, method='min')

# Identify users who were in the top 20 on at least one date.
# This creates a boolean Series where True means the user was in the top 20 at least once.
users_in_top_15_at_least_once = (ranked_df <= 20).any(axis=1)

# Get the list of users (their index labels) who meet the criteria.
users_to_keep = users_in_top_15_at_least_once[users_in_top_15_at_least_once].index

# --- Step 3: Filter the DataFrame ---

# Create a new DataFrame containing only the users who were in the top 20 at least once.
filtered_users_df = pivot_data_cleaned_transposed.loc[users_to_keep]

print("Original DataFrame shape:", pivot_data_cleaned_transposed.shape)
print("Filtered DataFrame shape:", filtered_users_df.shape)

Original DataFrame shape: (10573, 564)
Filtered DataFrame shape: (96, 564)


In [19]:
filtered_users_df.head()

day,user,2024-05-01,2024-05-02,2024-05-03,2024-05-04,2024-05-05,2024-05-06,2024-05-07,2024-05-09,2024-05-10,...,2026-02-27,2026-02-28,2026-03-01,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09
39,1206paul_,70.0,152.0,240.0,247.0,340.0,367.0,371.0,391.0,441.0,...,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0
64,1SKELTON,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.0,169.0,...,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0
332,Aloddin,24.0,46.0,48.0,70.0,77.0,126.0,135.0,140.0,142.0,...,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0
335,Aluminiumminimumimmunity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21658.0,21725.0,21806.0,21875.0,21889.0,21956.0,22029.0,22079.0,22132.0,22180.0
463,BanjoCash,11.0,63.0,65.0,65.0,65.0,65.0,65.0,65.0,65.0,...,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0


In [20]:
filtered_users_df.info

<bound method DataFrame.info of day                        user  2024-05-01  2024-05-02  2024-05-03  \
39                    1206paul_        70.0       152.0       240.0   
64                     1SKELTON         0.0         0.0         0.0   
332                     Aloddin        24.0        46.0        48.0   
335    Aluminiumminimumimmunity         0.0         0.0         0.0   
463                   BanjoCash        11.0        63.0        65.0   
...                         ...         ...         ...         ...   
10568                      빵댕이ㅋ         0.0         0.0         0.0   
10569                  쌍베님사랑합니다         0.0         0.0         0.0   
10570                     안톤958         0.0         0.0         0.0   
10571                    알래스카해달         0.0         0.0         0.0   
10572                     엘레레레ㅔ         0.0         0.0         0.0   

day    2024-05-04  2024-05-05  2024-05-06  2024-05-07  2024-05-09  2024-05-10  \
39          247.0       340.0     

In [21]:
filtered_users_df.head()

day,user,2024-05-01,2024-05-02,2024-05-03,2024-05-04,2024-05-05,2024-05-06,2024-05-07,2024-05-09,2024-05-10,...,2026-02-27,2026-02-28,2026-03-01,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09
39,1206paul_,70.0,152.0,240.0,247.0,340.0,367.0,371.0,391.0,441.0,...,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0,24752.0
64,1SKELTON,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.0,169.0,...,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0,4870.0
332,Aloddin,24.0,46.0,48.0,70.0,77.0,126.0,135.0,140.0,142.0,...,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0,8376.0
335,Aluminiumminimumimmunity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,21658.0,21725.0,21806.0,21875.0,21889.0,21956.0,22029.0,22079.0,22132.0,22180.0
463,BanjoCash,11.0,63.0,65.0,65.0,65.0,65.0,65.0,65.0,65.0,...,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0,448.0


In [22]:
filtered_users_df.to_excel('chattersRaceFULL.xlsx', sheet_name='Pivot Table')

In [23]:
data.shape

(3201298, 6)